# Subject: cpython
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Tools/c-analyzer/cpython

### File: __main__.py

In [ ]:
import logging
import sys
import textwrap

from c_common.scriptutil import (
    VERBOSITY,
    add_verbosity_cli,
    add_traceback_cli,
    add_commands_cli,
    add_kind_filtering_cli,
    add_files_cli,
    add_progress_cli,
    process_args_by_key,
    configure_logger,
    get_prog,
)
from c_parser.info import KIND
import c_parser.__main__ as c_parser
import c_analyzer.__main__ as c_analyzer
import c_analyzer as _c_analyzer
from c_analyzer.info import UNKNOWN
from . import _analyzer, _builtin_types, _capi, _files, _parser, REPO_ROOT


logger = logging.getLogger(__name__)


CHECK_EXPLANATION = textwrap.dedent('''
    -------------------------

    Non-constant global variables are generally not supported
    in the CPython repo.  We use a tool to analyze the C code
    and report if any unsupported globals are found.  The tool
    may be run manually with:

      ./python Tools/c-analyzer/check-c-globals.py --format summary [FILE]

    Occasionally the tool is unable to parse updated code.
    If this happens then add the file to the "EXCLUDED" list
    in Tools/c-analyzer/cpython/_parser.py and create a new
    issue for fixing the tool (and CC ericsnowcurrently
    on the issue).

    If the tool reports an unsupported global variable and
    it is actually const (and thus supported) then first try
    fixing the declaration appropriately in the code.  If that
    doesn't work then add the variable to the "should be const"
    section of Tools/c-analyzer/cpython/ignored.tsv.

    If the tool otherwise reports an unsupported global variable
    then first try to make it non-global, possibly adding to
    PyInterpreterState (for core code) or module state (for
    extension modules).  In an emergency, you can add the
    variable to Tools/c-analyzer/cpython/globals-to-fix.tsv
    to get CI passing, but doing so should be avoided.  If
    this course it taken, be sure to create an issue for
    eliminating the global (and CC ericsnowcurrently).
''')


def _resolve_filenames(filenames):
    if filenames:
        resolved = (_files.resolve_filename(f) for f in filenames)
    else:
        resolved = _files.iter_filenames()
    return resolved


#######################################
# the formats

def fmt_summary(analysis):
    # XXX Support sorting and grouping.
    supported = []
    unsupported = []
    for item in analysis:
        if item.supported:
            supported.append(item)
        else:
            unsupported.append(item)
    total = 0

    def section(name, groupitems):
        nonlocal total
        items, render = c_analyzer.build_section(name, groupitems,
                                                 relroot=REPO_ROOT)
        yield from render()
        total += len(items)

    yield ''
    yield '===================='
    yield 'supported'
    yield '===================='

    yield from section('types', supported)
    yield from section('variables', supported)

    yield ''
    yield '===================='
    yield 'unsupported'
    yield '===================='

    yield from section('types', unsupported)
    yield from section('variables', unsupported)

    yield ''
    yield f'grand total: {total}'


#######################################
# the checks

CHECKS = dict(c_analyzer.CHECKS, **{
    'globals': _analyzer.check_globals,
})

#######################################
# the commands

FILES_KWARGS = dict(excluded=_parser.EXCLUDED, nargs='*')


def _cli_parse(parser):
    process_output = c_parser.add_output_cli(parser)
    process_kind = add_kind_filtering_cli(parser)
    process_preprocessor = c_parser.add_preprocessor_cli(
        parser,
        get_preprocessor=_parser.get_preprocessor,
    )
    process_files = add_files_cli(parser, **FILES_KWARGS)
    return [
        process_output,
        process_kind,
        process_preprocessor,
        process_files,
    ]


def cmd_parse(filenames=None, **kwargs):
    filenames = _resolve_filenames(filenames)
    if 'get_file_preprocessor' not in kwargs:
        kwargs['get_file_preprocessor'] = _parser.get_preprocessor()
    c_parser.cmd_parse(
        filenames,
        relroot=REPO_ROOT,
        file_maxsizes=_parser.MAX_SIZES,
        **kwargs
    )


def _cli_check(parser, **kwargs):
    return c_analyzer._cli_check(parser, CHECKS, **kwargs, **FILES_KWARGS)


def cmd_check(filenames=None, **kwargs):
    filenames = _resolve_filenames(filenames)
    kwargs['get_file_preprocessor'] = _parser.get_preprocessor(log_err=print)
    try:
        c_analyzer.cmd_check(
            filenames,
            relroot=REPO_ROOT,
            _analyze=_analyzer.analyze,
            _CHECKS=CHECKS,
            file_maxsizes=_parser.MAX_SIZES,
            **kwargs
        )
    except SystemExit as exc:
        num_failed = exc.args[0] if getattr(exc, 'args', None) else None
        if isinstance(num_failed, int):
            if num_failed > 0:
                sys.stderr.flush()
                print(CHECK_EXPLANATION, flush=True)
        raise  # re-raise
    except Exception:
        sys.stderr.flush()
        print(CHECK_EXPLANATION, flush=True)
        raise  # re-raise


def cmd_analyze(filenames=None, **kwargs):
    formats = dict(c_analyzer.FORMATS)
    formats['summary'] = fmt_summary
    filenames = _resolve_filenames(filenames)
    kwargs['get_file_preprocessor'] = _parser.get_preprocessor(log_err=print)
    c_analyzer.cmd_analyze(
        filenames,
        relroot=REPO_ROOT,
        _analyze=_analyzer.analyze,
        formats=formats,
        file_maxsizes=_parser.MAX_SIZES,
        **kwargs
    )


def _cli_data(parser):
    filenames = False
    known = True
    return c_analyzer._cli_data(parser, filenames, known)


def cmd_data(datacmd, **kwargs):
    formats = dict(c_analyzer.FORMATS)
    formats['summary'] = fmt_summary
    filenames = (file
                 for file in _resolve_filenames(None)
                 if file not in _parser.EXCLUDED)
    kwargs['get_file_preprocessor'] = _parser.get_preprocessor(log_err=print)
    if datacmd == 'show':
        types = _analyzer.read_known()
        results = []
        for decl, info in types.items():
            if info is UNKNOWN:
                if decl.kind in (KIND.STRUCT, KIND.UNION):
                    extra = {'unsupported': ['type unknown'] * len(decl.members)}
                else:
                    extra = {'unsupported': ['type unknown']}
                info = (info, extra)
            results.append((decl, info))
            if decl.shortkey == 'struct _object':
                tempinfo = info
        known = _analyzer.Analysis.from_results(results)
        analyze = None
    elif datacmd == 'dump':
        known = _analyzer.KNOWN_FILE
        def analyze(files, **kwargs):
            decls = []
            for decl in _analyzer.iter_decls(files, **kwargs):
                if not KIND.is_type_decl(decl.kind):
                    continue
                if not decl.filename.endswith('.h'):
                    if decl.shortkey not in _analyzer.KNOWN_IN_DOT_C:
                        continue
                decls.append(decl)
            results = _c_analyzer.analyze_decls(
                decls,
                known={},
                analyze_resolved=_analyzer.analyze_resolved,
            )
            return _analyzer.Analysis.from_results(results)
    else:  # check
        known = _analyzer.read_known()
        def analyze(files, **kwargs):
            return _analyzer.iter_decls(files, **kwargs)
    extracolumns = None
    c_analyzer.cmd_data(
        datacmd,
        filenames,
        known,
        _analyze=analyze,
        formats=formats,
        extracolumns=extracolumns,
        relroot=REPO_ROOT,
        **kwargs
    )


def _cli_capi(parser):
    parser.add_argument('--levels', action='append', metavar='LEVEL[,...]')
    parser.add_argument(f'--public', dest='levels',
                        action='append_const', const='public')
    parser.add_argument(f'--no-public', dest='levels',
                        action='append_const', const='no-public')
    for level in _capi.LEVELS:
        parser.add_argument(f'--{level}', dest='levels',
                            action='append_const', const=level)
    def process_levels(args, *, argv=None):
        levels = []
        for raw in args.levels or ():
            for level in raw.replace(',', ' ').strip().split():
                if level == 'public':
                    levels.append('stable')
                    levels.append('cpython')
                elif level == 'no-public':
                    levels.append('private')
                    levels.append('internal')
                elif level in _capi.LEVELS:
                    levels.append(level)
                else:
                    parser.error(f'expected LEVEL to be one of {sorted(_capi.LEVELS)}, got {level!r}')
        args.levels = set(levels)

    parser.add_argument('--kinds', action='append', metavar='KIND[,...]')
    for kind in _capi.KINDS:
        parser.add_argument(f'--{kind}', dest='kinds',
                            action='append_const', const=kind)
    def process_kinds(args, *, argv=None):
        kinds = []
        for raw in args.kinds or ():
            for kind in raw.replace(',', ' ').strip().split():
                if kind in _capi.KINDS:
                    kinds.append(kind)
                else:
                    parser.error(f'expected KIND to be one of {sorted(_capi.KINDS)}, got {kind!r}')
        args.kinds = set(kinds)

    parser.add_argument('--group-by', dest='groupby',
                        choices=['level', 'kind'])

    parser.add_argument('--format', default='table')
    parser.add_argument('--summary', dest='format',
                        action='store_const', const='summary')
    def process_format(args, *, argv=None):
        orig = args.format
        args.format = _capi.resolve_format(args.format)
        if isinstance(args.format, str):
            if args.format not in _capi._FORMATS:
                parser.error(f'unsupported format {orig!r}')

    parser.add_argument('--show-empty', dest='showempty', action='store_true')
    parser.add_argument('--no-show-empty', dest='showempty', action='store_false')
    parser.set_defaults(showempty=None)

    # XXX Add --sort-by, --sort and --no-sort.

    parser.add_argument('--ignore', dest='ignored', action='append')
    def process_ignored(args, *, argv=None):
        ignored = []
        for raw in args.ignored or ():
            ignored.extend(raw.replace(',', ' ').strip().split())
        args.ignored = ignored or None

    parser.add_argument('filenames', nargs='*', metavar='FILENAME')
    process_progress = add_progress_cli(parser)

    return [
        process_levels,
        process_kinds,
        process_format,
        process_ignored,
        process_progress,
    ]


def cmd_capi(filenames=None, *,
             levels=None,
             kinds=None,
             groupby='kind',
             format='table',
             showempty=None,
             ignored=None,
             track_progress=None,
             verbosity=VERBOSITY,
             **kwargs
             ):
    render = _capi.get_renderer(format)

    filenames = _files.iter_header_files(filenames, levels=levels)
    #filenames = (file for file, _ in main_for_filenames(filenames))
    if track_progress:
        filenames = track_progress(filenames)
    items = _capi.iter_capi(filenames)
    if levels:
        items = (item for item in items if item.level in levels)
    if kinds:
        items = (item for item in items if item.kind in kinds)

    filter = _capi.resolve_filter(ignored)
    if filter:
        items = (item for item in items if filter(item, log=lambda msg: logger.log(1, msg)))

    lines = render(
        items,
        groupby=groupby,
        showempty=showempty,
        verbose=verbosity > VERBOSITY,
    )
    print()
    for line in lines:
        print(line)


def _cli_builtin_types(parser):
    parser.add_argument('--format', dest='fmt', default='table')
#    parser.add_argument('--summary', dest='format',
#                        action='store_const', const='summary')
    def process_format(args, *, argv=None):
        orig = args.fmt
        args.fmt = _builtin_types.resolve_format(args.fmt)
        if isinstance(args.fmt, str):
            if args.fmt not in _builtin_types._FORMATS:
                parser.error(f'unsupported format {orig!r}')

    parser.add_argument('--include-modules', dest='showmodules',
                        action='store_true')
    def process_modules(args, *, argv=None):
        pass

    return [
        process_format,
        process_modules,
    ]


def cmd_builtin_types(fmt, *,
                      showmodules=False,
                      verbosity=VERBOSITY,
                      ):
    render = _builtin_types.get_renderer(fmt)
    types = _builtin_types.iter_builtin_types()
    match = _builtin_types.resolve_matcher(showmodules)
    if match:
        types = (t for t in types if match(t, log=lambda msg: logger.log(1, msg)))

    lines = render(
        types,
#        verbose=verbosity > VERBOSITY,
    )
    print()
    for line in lines:
        print(line)


# We do not define any other cmd_*() handlers here,
# favoring those defined elsewhere.

COMMANDS = {
    'check': (
        'analyze and fail if the CPython source code has any problems',
        [_cli_check],
        cmd_check,
    ),
    'analyze': (
        'report on the state of the CPython source code',
        [(lambda p: c_analyzer._cli_analyze(p, **FILES_KWARGS))],
        cmd_analyze,
    ),
    'parse': (
        'parse the CPython source files',
        [_cli_parse],
        cmd_parse,
    ),
    'data': (
        'check/manage local data (e.g. known types, ignored vars, caches)',
        [_cli_data],
        cmd_data,
    ),
    'capi': (
        'inspect the C-API',
        [_cli_capi],
        cmd_capi,
    ),
    'builtin-types': (
        'show the builtin types',
        [_cli_builtin_types],
        cmd_builtin_types,
    ),
}


#######################################
# the script

def parse_args(argv=sys.argv[1:], prog=None, *, subset=None):
    import argparse
    parser = argparse.ArgumentParser(
        prog=prog or get_prog(),
    )

#    if subset == 'check' or subset == ['check']:
#        if checks is not None:
#            commands = dict(COMMANDS)
#            commands['check'] = list(commands['check'])
#            cli = commands['check'][1][0]
#            commands['check'][1][0] = (lambda p: cli(p, checks=checks))
    processors = add_commands_cli(
        parser,
        commands=COMMANDS,
        commonspecs=[
            add_verbosity_cli,
            add_traceback_cli,
        ],
        subset=subset,
    )

    args = parser.parse_args(argv)
    ns = vars(args)

    cmd = ns.pop('cmd')

    verbosity, traceback_cm = process_args_by_key(
        args,
        argv,
        processors[cmd],
        ['verbosity', 'traceback_cm'],
    )
    if cmd != 'parse':
        # "verbosity" is sent to the commands, so we put it back.
        args.verbosity = verbosity

    return cmd, ns, verbosity, traceback_cm


def main(cmd, cmd_kwargs):
    try:
        run_cmd = COMMANDS[cmd][-1]
    except KeyError:
        raise ValueError(f'unsupported cmd {cmd!r}')
    run_cmd(**cmd_kwargs)


if __name__ == '__main__':
    cmd, cmd_kwargs, verbosity, traceback_cm = parse_args()
    configure_logger(verbosity)
    with traceback_cm:
        main(cmd, cmd_kwargs)

### File: _analyzer.py

In [ ]:
import os.path

from c_common.clsutil import classonly
from c_parser.info import (
    KIND,
    Declaration,
    TypeDeclaration,
    Member,
    FIXED_TYPE,
)
from c_parser.match import (
    is_pots,
    is_funcptr,
)
from c_analyzer.match import (
    is_system_type,
    is_process_global,
    is_fixed_type,
    is_immutable,
)
import c_analyzer as _c_analyzer
import c_analyzer.info as _info
import c_analyzer.datafiles as _datafiles
from . import _parser, REPO_ROOT


_DATA_DIR = os.path.dirname(__file__)
KNOWN_FILE = os.path.join(_DATA_DIR, 'known.tsv')
IGNORED_FILE = os.path.join(_DATA_DIR, 'ignored.tsv')
NEED_FIX_FILE = os.path.join(_DATA_DIR, 'globals-to-fix.tsv')
KNOWN_IN_DOT_C = {
    'struct _odictobject': False,
    'PyTupleObject': False,
    'struct _typeobject': False,
    'struct _arena': True,  # ???
    'struct _frame': False,
    'struct _ts': True,  # ???
    'struct PyCodeObject': False,
    'struct _is': True,  # ???
    'PyWideStringList': True,  # ???
    # recursive
    'struct _dictkeysobject': False,
}
# These are loaded from the respective .tsv files upon first use.
_KNOWN = {
    # {(file, ID) | ID => info | bool}
    #'PyWideStringList': True,
}
#_KNOWN = {(Struct(None, typeid.partition(' ')[-1], None)
#           if typeid.startswith('struct ')
#           else TypeDef(None, typeid, None)
#           ): ([], {'unsupported': None if supported else True})
#          for typeid, supported in _KNOWN_IN_DOT_C.items()}
_IGNORED = {
    # {ID => reason}
}

# XXX We should be handling these through known.tsv.
_OTHER_SUPPORTED_TYPES = {
    # Holds tuple of strings, which we statically initialize:
    '_PyArg_Parser',
    # Uses of these should be const, but we don't worry about it.
    'PyModuleDef',
    'PyModuleDef_Slot[]',
    'PyType_Spec',
    'PyType_Slot[]',
    'PyMethodDef',
    'PyMethodDef[]',
    'PyMemberDef[]',
    'PyGetSetDef',
    'PyGetSetDef[]',
    'PyNumberMethods',
    'PySequenceMethods',
    'PyMappingMethods',
    'PyAsyncMethods',
    'PyBufferProcs',
    'PyStructSequence_Field[]',
    'PyStructSequence_Desc',
}

# XXX We should normalize all cases to a single name,
# e.g. "kwlist" (currently the most common).
_KWLIST_VARIANTS = [
    ('*', 'kwlist'),
    ('*', 'keywords'),
    ('*', 'kwargs'),
    ('Modules/_csv.c', 'dialect_kws'),
    ('Modules/_datetimemodule.c', 'date_kws'),
    ('Modules/_datetimemodule.c', 'datetime_kws'),
    ('Modules/_datetimemodule.c', 'time_kws'),
    ('Modules/_datetimemodule.c', 'timezone_kws'),
    ('Modules/_lzmamodule.c', 'optnames'),
    ('Modules/_lzmamodule.c', 'arg_names'),
    ('Modules/cjkcodecs/multibytecodec.c', 'incnewkwarglist'),
    ('Modules/cjkcodecs/multibytecodec.c', 'streamkwarglist'),
    ('Modules/socketmodule.c', 'kwnames'),
]

KINDS = frozenset((*KIND.TYPES, KIND.VARIABLE))


def read_known():
    if not _KNOWN:
        # Cache a copy the first time.
        extracols = None  # XXX
        #extracols = ['unsupported']
        known = _datafiles.read_known(KNOWN_FILE, extracols, REPO_ROOT)
        # For now we ignore known.values() (i.e. "extra").
        types, _ = _datafiles.analyze_known(
            known,
            analyze_resolved=analyze_resolved,
        )
        _KNOWN.update(types)
    return _KNOWN.copy()


def write_known():
    raise NotImplementedError
    datafiles.write_known(decls, IGNORED_FILE, ['unsupported'], relroot=REPO_ROOT)


def read_ignored():
    if not _IGNORED:
        _IGNORED.update(_datafiles.read_ignored(IGNORED_FILE, relroot=REPO_ROOT))
        _IGNORED.update(_datafiles.read_ignored(NEED_FIX_FILE, relroot=REPO_ROOT))
    return dict(_IGNORED)


def write_ignored():
    raise NotImplementedError
    _datafiles.write_ignored(variables, IGNORED_FILE, relroot=REPO_ROOT)


def analyze(filenames, *,
            skip_objects=False,
            **kwargs
            ):
    if skip_objects:
        # XXX Set up a filter.
        raise NotImplementedError

    known = read_known()

    decls = iter_decls(filenames)
    results = _c_analyzer.analyze_decls(
        decls,
        known,
        analyze_resolved=analyze_resolved,
    )
    analysis = Analysis.from_results(results)

    return analysis


def iter_decls(filenames, **kwargs):
    decls = _c_analyzer.iter_decls(
        filenames,
        # We ignore functions (and statements).
        kinds=KINDS,
        parse_files=_parser.parse_files,
        **kwargs
    )
    for decl in decls:
        if not decl.data:
            # Ignore forward declarations.
            continue
        yield decl


def analyze_resolved(resolved, decl, types, knowntypes, extra=None):
    if decl.kind not in KINDS:
        # Skip it!
        return None

    typedeps = resolved
    if typedeps is _info.UNKNOWN:
        if decl.kind in (KIND.STRUCT, KIND.UNION):
            typedeps = [typedeps] * len(decl.members)
        else:
            typedeps = [typedeps]
    #assert isinstance(typedeps, (list, TypeDeclaration)), typedeps

    if extra is None:
        extra = {}
    elif 'unsupported' in extra:
        raise NotImplementedError((decl, extra))

    unsupported = _check_unsupported(decl, typedeps, types, knowntypes)
    extra['unsupported'] = unsupported

    return typedeps, extra


def _check_unsupported(decl, typedeps, types, knowntypes):
    if typedeps is None:
        raise NotImplementedError(decl)

    if decl.kind in (KIND.STRUCT, KIND.UNION):
        return _check_members(decl, typedeps, types, knowntypes)
    elif decl.kind is KIND.ENUM:
        if typedeps:
            raise NotImplementedError((decl, typedeps))
        return None
    else:
        return _check_typedep(decl, typedeps, types, knowntypes)


def _check_members(decl, typedeps, types, knowntypes):
    if isinstance(typedeps, TypeDeclaration):
        raise NotImplementedError((decl, typedeps))

    #members = decl.members or ()  # A forward decl has no members.
    members = decl.members
    if not members:
        # A forward decl has no members, but that shouldn't surface here..
        raise NotImplementedError(decl)
    if len(members) != len(typedeps):
        raise NotImplementedError((decl, typedeps))

    unsupported = []
    for member, typedecl in zip(members, typedeps):
        checked = _check_typedep(member, typedecl, types, knowntypes)
        unsupported.append(checked)
    if any(None if v is FIXED_TYPE else v for v in unsupported):
        return unsupported
    elif FIXED_TYPE in unsupported:
        return FIXED_TYPE
    else:
        return None


def _check_typedep(decl, typedecl, types, knowntypes):
    if not isinstance(typedecl, TypeDeclaration):
        if hasattr(type(typedecl), '__len__'):
            if len(typedecl) == 1:
                typedecl, = typedecl
    if typedecl is None:
        # XXX Fail?
        return 'typespec (missing)'
    elif typedecl is _info.UNKNOWN:
        if _has_other_supported_type(decl):
            return None
        # XXX Is this right?
        return 'typespec (unknown)'
    elif not isinstance(typedecl, TypeDeclaration):
        raise NotImplementedError((decl, typedecl))

    if isinstance(decl, Member):
        return _check_vartype(decl, typedecl, types, knowntypes)
    elif not isinstance(decl, Declaration):
        raise NotImplementedError(decl)
    elif decl.kind is KIND.TYPEDEF:
        return _check_vartype(decl, typedecl, types, knowntypes)
    elif decl.kind is KIND.VARIABLE:
        if not is_process_global(decl):
            return None
        if _is_kwlist(decl):
            return None
        if _has_other_supported_type(decl):
            return None
        checked = _check_vartype(decl, typedecl, types, knowntypes)
        return 'mutable' if checked is FIXED_TYPE else checked
    else:
        raise NotImplementedError(decl)


def _is_kwlist(decl):
    # keywords for PyArg_ParseTupleAndKeywords()
    # "static char *name[]" -> "static const char * const name[]"
    # XXX These should be made const.
    for relpath, name in _KWLIST_VARIANTS:
        if decl.name == name:
            if relpath == '*':
                break
            assert os.path.isabs(decl.file.filename)
            relpath = os.path.normpath(relpath)
            if decl.file.filename.endswith(os.path.sep + relpath):
                break
    else:
        return False
    vartype = ''.join(str(decl.vartype).split())
    return vartype == 'char*[]'

def _is_local_static_mutex(decl):
    if not hasattr(decl, "vartype"):
        return False

    if not hasattr(decl, "parent") or decl.parent is None:
        # We only want to allow local variables
        return False

    vartype = decl.vartype
    return (vartype.typespec == 'PyMutex') and (decl.storage == 'static')

def _has_other_supported_type(decl):
    if hasattr(decl, 'file') and decl.file.filename.endswith('.c.h'):
        assert 'clinic' in decl.file.filename, (decl,)
        if decl.name == '_kwtuple':
            return True
    if _is_local_static_mutex(decl):
        # GH-127081: Local static mutexes are used to
        # wrap libc functions that aren't thread safe
        return True
    vartype = str(decl.vartype).split()
    if vartype[0] == 'struct':
        vartype = vartype[1:]
    vartype = ''.join(vartype)
    return vartype in _OTHER_SUPPORTED_TYPES


def _check_vartype(decl, typedecl, types, knowntypes):
    """Return failure reason."""
    checked = _check_typespec(decl, typedecl, types, knowntypes)
    if checked:
        return checked
    if is_immutable(decl.vartype):
        return None
    if is_fixed_type(decl.vartype):
        return FIXED_TYPE
    return 'mutable'


def _check_typespec(decl, typedecl, types, knowntypes):
    typespec = decl.vartype.typespec
    if typedecl is not None:
        found = types.get(typedecl)
        if found is None:
            found = knowntypes.get(typedecl)

        if found is not None:
            _, extra = found
            if extra is None:
                # XXX Under what circumstances does this happen?
                extra = {}
            unsupported = extra.get('unsupported')
            if unsupported is FIXED_TYPE:
                unsupported = None
            return 'typespec' if unsupported else None
    # Fall back to default known types.
    if is_pots(typespec):
        return None
    elif is_system_type(typespec):
        return None
    elif is_funcptr(decl.vartype):
        return None
    return 'typespec'


class Analyzed(_info.Analyzed):

    @classonly
    def is_target(cls, raw):
        if not super().is_target(raw):
            return False
        if raw.kind not in KINDS:
            return False
        return True

    #@classonly
    #def _parse_raw_result(cls, result, extra):
    #    typedecl, extra = super()._parse_raw_result(result, extra)
    #    if typedecl is None:
    #        return None, extra
    #    raise NotImplementedError

    def __init__(self, item, typedecl=None, *, unsupported=None, **extra):
        if 'unsupported' in extra:
            raise NotImplementedError((item, typedecl, unsupported, extra))
        if not unsupported:
            unsupported = None
        elif isinstance(unsupported, (str, TypeDeclaration)):
            unsupported = (unsupported,)
        elif unsupported is not FIXED_TYPE:
            unsupported = tuple(unsupported)
        self.unsupported = unsupported
        extra['unsupported'] = self.unsupported  # ...for __repr__(), etc.
        if self.unsupported is None:
            #self.supported = None
            self.supported = True
        elif self.unsupported is FIXED_TYPE:
            if item.kind is KIND.VARIABLE:
                raise NotImplementedError(item, typedecl, unsupported)
            self.supported = True
        else:
            self.supported = not self.unsupported
        super().__init__(item, typedecl, **extra)

    def render(self, fmt='line', *, itemonly=False):
        if fmt == 'raw':
            yield repr(self)
            return
        rendered = super().render(fmt, itemonly=itemonly)
        # XXX ???
        #if itemonly:
        #    yield from rendered
        supported = self.supported
        if fmt in ('line', 'brief'):
            rendered, = rendered
            parts = [
                '+' if supported else '-' if supported is False else '',
                rendered,
            ]
            yield '\t'.join(parts)
        elif fmt == 'summary':
            raise NotImplementedError(fmt)
        elif fmt == 'full':
            yield from rendered
            if supported:
                yield f'\tsupported:\t{supported}'
        else:
            raise NotImplementedError(fmt)


class Analysis(_info.Analysis):
    _item_class = Analyzed

    @classonly
    def build_item(cls, info, result=None):
        if not isinstance(info, Declaration) or info.kind not in KINDS:
            raise NotImplementedError((info, result))
        return super().build_item(info, result)


def check_globals(analysis):
    # yield (data, failure)
    ignored = read_ignored()
    for item in analysis:
        if item.kind != KIND.VARIABLE:
            continue
        if item.supported:
            continue
        if item.id in ignored:
            continue
        reason = item.unsupported
        if not reason:
            reason = '???'
        elif not isinstance(reason, str):
            if len(reason) == 1:
                reason, = reason
        reason = f'({reason})'
        yield item, f'not supported {reason:20}\t{item.storage or ""} {item.vartype}'

### File: _builtin_types.py

In [ ]:
from collections import namedtuple
import os.path
import re
import textwrap

from c_common import tables
from . import REPO_ROOT
from ._files import iter_header_files, iter_filenames


CAPI_PREFIX = os.path.join('Include', '')
INTERNAL_PREFIX = os.path.join('Include', 'internal', '')

REGEX = re.compile(textwrap.dedent(rf'''
    (?:
        ^
        (?:
            (?:
                (?:
                    (?:
                        (?:
                            ( static )  # <static>
                            \s+
                            |
                            ( extern )  # <extern>
                            \s+
                         )?
                        PyTypeObject \s+
                     )
                    |
                    (?:
                        ( PyAPI_DATA )  # <capi>
                        \s* [(] \s* PyTypeObject \s* [)] \s*
                     )
                 )
                (\w+)  # <name>
                \s*
                (?:
                    (?:
                        ( = \s* {{ )  # <def>
                        $
                     )
                    |
                    ( ; )  # <decl>
                 )
             )
            |
            (?:
                # These are specific to Objects/exceptions.c:
                (?:
                    SimpleExtendsException
                    |
                    MiddlingExtendsException
                    |
                    ComplexExtendsException
                 )
                \( \w+ \s* , \s*
                ( \w+ )  # <excname>
                \s* ,
             )
         )
    )
'''), re.VERBOSE)


def _parse_line(line):
    m = re.match(REGEX, line)
    if not m:
        return None
    (static, extern, capi,
     name,
     def_, decl,
     excname,
     ) = m.groups()
    if def_:
        isdecl = False
        if extern or capi:
            raise NotImplementedError(line)
        kind = 'static' if static else None
    elif excname:
        name = f'_PyExc_{excname}'
        isdecl = False
        kind = 'static'
    else:
        isdecl = True
        if static:
            kind = 'static'
        elif extern:
            kind = 'extern'
        elif capi:
            kind = 'capi'
        else:
            kind = None
    return name, isdecl, kind


class BuiltinTypeDecl(namedtuple('BuiltinTypeDecl', 'file lno name kind')):

    KINDS = {
        'static',
        'extern',
        'capi',
        'forward',
    }

    @classmethod
    def from_line(cls, line, filename, lno):
        # This is similar to ._capi.CAPIItem.from_line().
        parsed = _parse_line(line)
        if not parsed:
            return None
        name, isdecl, kind = parsed
        if not isdecl:
            return None
        return cls.from_parsed(name, kind, filename, lno)

    @classmethod
    def from_parsed(cls, name, kind, filename, lno):
        if not kind:
            kind = 'forward'
        return cls.from_values(filename, lno, name, kind)

    @classmethod
    def from_values(cls, filename, lno, name, kind):
        if kind not in cls.KINDS:
            raise ValueError(f'unsupported kind {kind!r}')
        self = cls(filename, lno, name, kind)
        if self.kind not in ('extern', 'capi') and self.api:
            raise NotImplementedError(self)
        elif self.kind == 'capi' and not self.api:
            raise NotImplementedError(self)
        return self

    @property
    def relfile(self):
        return self.file[len(REPO_ROOT) + 1:]

    @property
    def api(self):
        return self.relfile.startswith(CAPI_PREFIX)

    @property
    def internal(self):
        return self.relfile.startswith(INTERNAL_PREFIX)

    @property
    def private(self):
        if not self.name.startswith('_'):
            return False
        return self.api and not self.internal

    @property
    def public(self):
        if self.kind != 'capi':
            return False
        return not self.internal and not self.private


class BuiltinTypeInfo(namedtuple('BuiltinTypeInfo', 'file lno name static decl')):

    @classmethod
    def from_line(cls, line, filename, lno, *, decls=None):
        parsed = _parse_line(line)
        if not parsed:
            return None
        name, isdecl, kind = parsed
        if isdecl:
            return None
        return cls.from_parsed(name, kind, filename, lno, decls=decls)

    @classmethod
    def from_parsed(cls, name, kind, filename, lno, *, decls=None):
        if not kind:
            static = False
        elif kind == 'static':
            static = True
        else:
            raise NotImplementedError((filename, line, kind))
        decl = decls.get(name) if decls else None
        return cls(filename, lno, name, static, decl)

    @property
    def relfile(self):
        return self.file[len(REPO_ROOT) + 1:]

    @property
    def exported(self):
        return not self.static

    @property
    def api(self):
        if not self.decl:
            return False
        return self.decl.api

    @property
    def internal(self):
        if not self.decl:
            return False
        return self.decl.internal

    @property
    def private(self):
        if not self.decl:
            return False
        return self.decl.private

    @property
    def public(self):
        if not self.decl:
            return False
        return self.decl.public

    @property
    def inmodule(self):
        return self.relfile.startswith('Modules' + os.path.sep)

    def render_rowvalues(self, kinds):
        row = {
            'name': self.name,
            **{k: '' for k in kinds},
            'filename': f'{self.relfile}:{self.lno}',
        }
        if self.static:
            kind = 'static'
        else:
            if self.internal:
                kind = 'internal'
            elif self.private:
                kind = 'private'
            elif self.public:
                kind = 'public'
            else:
                kind = 'global'
        row['kind'] = kind
        row[kind] = kind
        return row


def _ensure_decl(decl, decls):
    prev = decls.get(decl.name)
    if prev:
        if decl.kind == 'forward':
            return None
        if prev.kind != 'forward':
            if decl.kind == prev.kind and decl.file == prev.file:
                assert decl.lno != prev.lno, (decl, prev)
                return None
            raise NotImplementedError(f'duplicate {decl} (was {prev}')
    decls[decl.name] = decl


def iter_builtin_types(filenames=None):
    decls = {}
    seen = set()
    for filename in iter_header_files():
        seen.add(filename)
        with open(filename) as infile:
            for lno, line in enumerate(infile, 1):
                decl = BuiltinTypeDecl.from_line(line, filename, lno)
                if not decl:
                    continue
                _ensure_decl(decl, decls)
    srcfiles = []
    for filename in iter_filenames():
        if filename.endswith('.c'):
            srcfiles.append(filename)
            continue
        if filename in seen:
            continue
        with open(filename) as infile:
            for lno, line in enumerate(infile, 1):
                decl = BuiltinTypeDecl.from_line(line, filename, lno)
                if not decl:
                    continue
                _ensure_decl(decl, decls)

    for filename in srcfiles:
        with open(filename) as infile:
            localdecls = {}
            for lno, line in enumerate(infile, 1):
                parsed = _parse_line(line)
                if not parsed:
                    continue
                name, isdecl, kind = parsed
                if isdecl:
                    decl = BuiltinTypeDecl.from_parsed(name, kind, filename, lno)
                    if not decl:
                        raise NotImplementedError((filename, line))
                    _ensure_decl(decl, localdecls)
                else:
                    builtin = BuiltinTypeInfo.from_parsed(
                            name, kind, filename, lno,
                            decls=decls if name in decls else localdecls)
                    if not builtin:
                        raise NotImplementedError((filename, line))
                    yield builtin


def resolve_matcher(showmodules=False):
    def match(info, *, log=None):
        if not info.inmodule:
            return True
        if log is not None:
            log(f'ignored {info.name!r}')
        return False
    return match


##################################
# CLI rendering

def resolve_format(fmt):
    if not fmt:
        return 'table'
    elif isinstance(fmt, str) and fmt in _FORMATS:
        return fmt
    else:
        raise NotImplementedError(fmt)


def get_renderer(fmt):
    fmt = resolve_format(fmt)
    if isinstance(fmt, str):
        try:
            return _FORMATS[fmt]
        except KeyError:
            raise ValueError(f'unsupported format {fmt!r}')
    else:
        raise NotImplementedError(fmt)


def render_table(types):
    types = sorted(types, key=(lambda t: t.name))
    colspecs = tables.resolve_columns(
            'name:<33 static:^ global:^ internal:^ private:^ public:^ filename:<30')
    header, div, rowfmt = tables.build_table(colspecs)
    leader = ' ' * sum(c.width+2 for c in colspecs[:3]) + '   '
    yield leader + f'{"API":^29}'
    yield leader + '-' * 29
    yield header
    yield div
    kinds = [c[0] for c in colspecs[1:-1]]
    counts = {k: 0 for k in kinds}
    base = {k: '' for k in kinds}
    for t in types:
        row = t.render_rowvalues(kinds)
        kind = row['kind']
        yield rowfmt.format(**row)
        counts[kind] += 1
    yield ''
    yield f'total: {sum(counts.values()):>3}'
    for kind in kinds:
        yield f'  {kind:>10}: {counts[kind]:>3}'


def render_repr(types):
    for t in types:
        yield repr(t)


_FORMATS = {
    'table': render_table,
    'repr': render_repr,
}

### File: _capi.py

In [ ]:
from collections import namedtuple
import logging
import os
import os.path
import re
import textwrap

from c_common.tables import build_table, resolve_columns
from c_parser.parser._regexes import _ind
from ._files import iter_header_files
from . import REPO_ROOT


logger = logging.getLogger(__name__)


INCLUDE_ROOT = os.path.join(REPO_ROOT, 'Include')
INCLUDE_CPYTHON = os.path.join(INCLUDE_ROOT, 'cpython')
INCLUDE_INTERNAL = os.path.join(INCLUDE_ROOT, 'internal')

_MAYBE_NESTED_PARENS = textwrap.dedent(r'''
    (?:
        (?: [^(]* [(] [^()]* [)] )* [^(]*
    )
''')

CAPI_FUNC = textwrap.dedent(rf'''
    (?:
        ^
        \s*
        PyAPI_FUNC \s*
        [(]
        {_ind(_MAYBE_NESTED_PARENS, 2)}
        [)] \s*
        (\w+)  # <func>
        \s* [(]
    )
''')
CAPI_DATA = textwrap.dedent(rf'''
    (?:
        ^
        \s*
        PyAPI_DATA \s*
        [(]
        {_ind(_MAYBE_NESTED_PARENS, 2)}
        [)] \s*
        (\w+)  # <data>
        \b [^(]
    )
''')
CAPI_INLINE = textwrap.dedent(r'''
    (?:
        ^
        \s*
        static \s+ inline \s+
        .*?
        \s+
        ( \w+ )  # <inline>
        \s* [(]
    )
''')
CAPI_MACRO = textwrap.dedent(r'''
    (?:
        (\w+)  # <macro>
        [(]
    )
''')
CAPI_CONSTANT = textwrap.dedent(r'''
    (?:
        (\w+)  # <constant>
        \s+ [^(]
    )
''')
CAPI_DEFINE = textwrap.dedent(rf'''
    (?:
        ^
        \s* [#] \s* define \s+
        (?:
            {_ind(CAPI_MACRO, 3)}
            |
            {_ind(CAPI_CONSTANT, 3)}
            |
            (?:
                # ignored
                \w+   # <defined_name>
                \s*
                $
            )
        )
    )
''')
CAPI_RE = re.compile(textwrap.dedent(rf'''
    (?:
        {_ind(CAPI_FUNC, 2)}
        |
        {_ind(CAPI_DATA, 2)}
        |
        {_ind(CAPI_INLINE, 2)}
        |
        {_ind(CAPI_DEFINE, 2)}
    )
'''), re.VERBOSE)

KINDS = [
    'func',
    'data',
    'inline',
    'macro',
    'constant',
]


def _parse_line(line, prev=None):
    last = line
    if prev:
        if not prev.endswith(os.linesep):
            prev += os.linesep
        line = prev + line
    m = CAPI_RE.match(line)
    if not m:
        if not prev and line.startswith('static inline '):
            return line  # the new "prev"
        #if 'PyAPI_' in line or '#define ' in line or ' define ' in line:
        #    print(line)
        return None
    results = zip(KINDS, m.groups())
    for kind, name in results:
        if name:
            clean = last.split('//')[0].rstrip()
            if clean.endswith('*/'):
                clean = clean.split('/*')[0].rstrip()

            if kind == 'macro' or kind == 'constant':
                if not clean.endswith('\\'):
                    return name, kind
            elif kind == 'inline':
                if clean.endswith('}'):
                    if not prev or clean == '}':
                        return name, kind
            elif kind == 'func' or kind == 'data':
                if clean.endswith(';'):
                    return name, kind
            else:
                # This should not be reached.
                raise NotImplementedError
            return line  # the new "prev"
    # It was a plain #define.
    return None


LEVELS = [
    'stable',
    'cpython',
    'private',
    'internal',
]

def _get_level(filename, name, *,
               _cpython=INCLUDE_CPYTHON + os.path.sep,
               _internal=INCLUDE_INTERNAL + os.path.sep,
               ):
    if filename.startswith(_internal):
        return 'internal'
    elif name.startswith('_'):
        return 'private'
    elif os.path.dirname(filename) == INCLUDE_ROOT:
        return 'stable'
    elif filename.startswith(_cpython):
        return 'cpython'
    else:
        raise NotImplementedError
    #return '???'


GROUPINGS = {
    'kind': KINDS,
    'level': LEVELS,
}


class CAPIItem(namedtuple('CAPIItem', 'file lno name kind level')):

    @classmethod
    def from_line(cls, line, filename, lno, prev=None):
        parsed = _parse_line(line, prev)
        if not parsed:
            return None, None
        if isinstance(parsed, str):
            # incomplete
            return None, parsed
        name, kind = parsed
        level = _get_level(filename, name)
        self = cls(filename, lno, name, kind, level)
        if prev:
            self._text = (prev + line).rstrip().splitlines()
        else:
            self._text = [line.rstrip()]
        return self, None

    @property
    def relfile(self):
        return self.file[len(REPO_ROOT) + 1:]

    @property
    def text(self):
        try:
            return self._text
        except AttributeError:
            # XXX Actually ready the text from disk?.
            self._text = []
            if self.kind == 'data':
                self._text = [
                    f'PyAPI_DATA(...) {self.name}',
                ]
            elif self.kind == 'func':
                self._text = [
                    f'PyAPI_FUNC(...) {self.name}(...);',
                ]
            elif self.kind == 'inline':
                self._text = [
                    f'static inline {self.name}(...);',
                ]
            elif self.kind == 'macro':
                self._text = [
                    f'#define {self.name}(...) \\',
                    f'    ...',
                ]
            elif self.kind == 'constant':
                self._text = [
                    f'#define {self.name} ...',
                ]
            else:
                raise NotImplementedError

            return self._text


def _parse_groupby(raw):
    if not raw:
        raw = 'kind'

    if isinstance(raw, str):
        groupby = raw.replace(',', ' ').strip().split()
    else:
        raise NotImplementedError

    if not all(v in GROUPINGS for v in groupby):
        raise ValueError(f'invalid groupby value {raw!r}')
    return groupby


def _resolve_full_groupby(groupby):
    if isinstance(groupby, str):
        groupby = [groupby]
    groupings = []
    for grouping in groupby + list(GROUPINGS):
        if grouping not in groupings:
            groupings.append(grouping)
    return groupings


def summarize(items, *, groupby='kind', includeempty=True, minimize=None):
    if minimize is None:
        if includeempty is None:
            minimize = True
            includeempty = False
        else:
            minimize = includeempty
    elif includeempty is None:
        includeempty = minimize
    elif minimize and includeempty:
        raise ValueError(f'cannot minimize and includeempty at the same time')

    groupby = _parse_groupby(groupby)[0]
    _outer, _inner = _resolve_full_groupby(groupby)
    outers = GROUPINGS[_outer]
    inners = GROUPINGS[_inner]

    summary = {
        'totals': {
            'all': 0,
            'subs': {o: 0 for o in outers},
            'bygroup': {o: {i: 0 for i in inners}
                        for o in outers},
        },
    }

    for item in items:
        outer = getattr(item, _outer)
        inner = getattr(item, _inner)
        # Update totals.
        summary['totals']['all'] += 1
        summary['totals']['subs'][outer] += 1
        summary['totals']['bygroup'][outer][inner] += 1

    if not includeempty:
        subtotals = summary['totals']['subs']
        bygroup = summary['totals']['bygroup']
        for outer in outers:
            if subtotals[outer] == 0:
                del subtotals[outer]
                del bygroup[outer]
                continue

            for inner in inners:
                if bygroup[outer][inner] == 0:
                    del bygroup[outer][inner]
            if minimize:
                if len(bygroup[outer]) == 1:
                    del bygroup[outer]

    return summary


def _parse_capi(lines, filename):
    if isinstance(lines, str):
        lines = lines.splitlines()
    prev = None
    for lno, line in enumerate(lines, 1):
        parsed, prev = CAPIItem.from_line(line, filename, lno, prev)
        if parsed:
            yield parsed
    if prev:
        parsed, prev = CAPIItem.from_line('', filename, lno, prev)
        if parsed:
            yield parsed
        if prev:
            print('incomplete match:')
            print(filename)
            print(prev)
            raise Exception


def iter_capi(filenames=None):
    for filename in iter_header_files(filenames):
        with open(filename) as infile:
            for item in _parse_capi(infile, filename):
                yield item


def resolve_filter(ignored):
    if not ignored:
        return None
    ignored = set(_resolve_ignored(ignored))
    def filter(item, *, log=None):
        if item.name not in ignored:
            return True
        if log is not None:
            log(f'ignored {item.name!r}')
        return False
    return filter


def _resolve_ignored(ignored):
    if isinstance(ignored, str):
        ignored = [ignored]
    for raw in ignored:
        if isinstance(raw, str):
            if raw.startswith('|'):
                yield raw[1:]
            elif raw.startswith('<') and raw.endswith('>'):
                filename = raw[1:-1]
                try:
                    infile = open(filename)
                except Exception as exc:
                    logger.error(f'ignore file failed: {exc}')
                    continue
                logger.log(1, f'reading ignored names from {filename!r}')
                with infile:
                    for line in infile:
                        if not line:
                            continue
                        if line[0].isspace():
                            continue
                        line = line.partition('#')[0].rstrip()
                        if line:
                            # XXX Recurse?
                            yield line
            else:
                raw = raw.strip()
                if raw:
                    yield raw
        else:
            raise NotImplementedError


def _collate(items, groupby, includeempty):
    groupby = _parse_groupby(groupby)[0]
    maxfilename = maxname = maxkind = maxlevel = 0

    collated = {}
    groups = GROUPINGS[groupby]
    for group in groups:
        collated[group] = []

    for item in items:
        key = getattr(item, groupby)
        collated[key].append(item)
        maxfilename = max(len(item.relfile), maxfilename)
        maxname = max(len(item.name), maxname)
        maxkind = max(len(item.kind), maxkind)
        maxlevel = max(len(item.level), maxlevel)
    if not includeempty:
        for group in groups:
            if not collated[group]:
                del collated[group]
    maxextra = {
        'kind': maxkind,
        'level': maxlevel,
    }
    return collated, groupby, maxfilename, maxname, maxextra


def _get_sortkey(sort, _groupby, _columns):
    if sort is True or sort is None:
        # For now:
        def sortkey(item):
            return (
                item.level == 'private',
                LEVELS.index(item.level),
                KINDS.index(item.kind),
                os.path.dirname(item.file),
                os.path.basename(item.file),
                item.name,
            )
        return sortkey

        sortfields = 'not-private level kind dirname basename name'.split()
    elif isinstance(sort, str):
        sortfields = sort.replace(',', ' ').strip().split()
    elif callable(sort):
        return sort
    else:
        raise NotImplementedError

    # XXX Build a sortkey func from sortfields.
    raise NotImplementedError


##################################
# CLI rendering

_MARKERS = {
    'level': {
        'S': 'stable',
        'C': 'cpython',
        'P': 'private',
        'I': 'internal',
    },
    'kind': {
        'F': 'func',
        'D': 'data',
        'I': 'inline',
        'M': 'macro',
        'C': 'constant',
    },
}


def resolve_format(format):
    if not format:
        return 'table'
    elif isinstance(format, str) and format in _FORMATS:
        return format
    else:
        return resolve_columns(format)


def get_renderer(format):
    format = resolve_format(format)
    if isinstance(format, str):
        try:
            return _FORMATS[format]
        except KeyError:
            raise ValueError(f'unsupported format {format!r}')
    else:
        def render(items, **kwargs):
            return render_table(items, columns=format, **kwargs)
        return render


def render_table(items, *,
                 columns=None,
                 groupby='kind',
                 sort=True,
                 showempty=False,
                 verbose=False,
                 ):
    if groupby is None:
        groupby = 'kind'
    if showempty is None:
        showempty = False

    if groupby:
        (collated, groupby, maxfilename, maxname, maxextra,
         ) = _collate(items, groupby, showempty)
        for grouping in GROUPINGS:
            maxextra[grouping] = max(len(g) for g in GROUPINGS[grouping])

        _, extra = _resolve_full_groupby(groupby)
        extras = [extra]
        markers = {extra: _MARKERS[extra]}

        groups = GROUPINGS[groupby]
    else:
        # XXX Support no grouping?
        raise NotImplementedError

    if columns:
        def get_extra(item):
            return {extra: getattr(item, extra)
                    for extra in ('kind', 'level')}
    else:
        if verbose:
            extracols = [f'{extra}:{maxextra[extra]}'
                         for extra in extras]
            def get_extra(item):
                return {extra: getattr(item, extra)
                        for extra in extras}
        elif len(extras) == 1:
            extra, = extras
            extracols = [f'{m}:1' for m in markers[extra]]
            def get_extra(item):
                return {m: m if getattr(item, extra) == markers[extra][m] else ''
                        for m in markers[extra]}
        else:
            raise NotImplementedError
            #extracols = [[f'{m}:1' for m in markers[extra]]
            #             for extra in extras]
            #def get_extra(item):
            #    values = {}
            #    for extra in extras:
            #        cur = markers[extra]
            #        for m in cur:
            #            values[m] = m if getattr(item, m) == cur[m] else ''
            #    return values
        columns = [
            f'filename:{maxfilename}',
            f'name:{maxname}',
            *extracols,
        ]
    header, div, fmt = build_table(columns)

    if sort:
        sortkey = _get_sortkey(sort, groupby, columns)

    total = 0
    for group, grouped in collated.items():
        if not showempty and group not in collated:
            continue
        yield ''
        yield f' === {group} ==='
        yield ''
        yield header
        yield div
        if grouped:
            if sort:
                grouped = sorted(grouped, key=sortkey)
            for item in grouped:
                yield fmt.format(
                    filename=item.relfile,
                    name=item.name,
                    **get_extra(item),
                )
        yield div
        subtotal = len(grouped)
        yield f'  sub-total: {subtotal}'
        total += subtotal
    yield ''
    yield f'total: {total}'


def render_full(items, *,
                groupby='kind',
                sort=None,
                showempty=None,
                verbose=False,
                ):
    if groupby is None:
        groupby = 'kind'
    if showempty is None:
        showempty = False

    if sort:
        sortkey = _get_sortkey(sort, groupby, None)

    if groupby:
        collated, groupby, _, _, _ = _collate(items, groupby, showempty)
        for group, grouped in collated.items():
            yield '#' * 25
            yield f'# {group} ({len(grouped)})'
            yield '#' * 25
            yield ''
            if not grouped:
                continue
            if sort:
                grouped = sorted(grouped, key=sortkey)
            for item in grouped:
                yield from _render_item_full(item, groupby, verbose)
                yield ''
    else:
        if sort:
            items = sorted(items, key=sortkey)
        for item in items:
            yield from _render_item_full(item, None, verbose)
            yield ''


def _render_item_full(item, groupby, verbose):
    yield item.name
    yield f'  {"filename:":10} {item.relfile}'
    for extra in ('kind', 'level'):
        yield f'  {extra+":":10} {getattr(item, extra)}'
    if verbose:
        print('  ---------------------------------------')
        for lno, line in enumerate(item.text, item.lno):
            print(f'  | {lno:3} {line}')
        print('  ---------------------------------------')


def render_summary(items, *,
                   groupby='kind',
                   sort=None,
                   showempty=None,
                   verbose=False,
                   ):
    if groupby is None:
        groupby = 'kind'
    summary = summarize(
        items,
        groupby=groupby,
        includeempty=showempty,
        minimize=None if showempty else not verbose,
    )

    subtotals = summary['totals']['subs']
    bygroup = summary['totals']['bygroup']
    for outer, subtotal in subtotals.items():
        if bygroup:
            subtotal = f'({subtotal})'
            yield f'{outer + ":":20} {subtotal:>8}'
        else:
            yield f'{outer + ":":10} {subtotal:>8}'
        if outer in bygroup:
            for inner, count in bygroup[outer].items():
                yield f'   {inner + ":":9} {count}'
    total = f'*{summary["totals"]["all"]}*'
    label = '*total*:'
    if bygroup:
        yield f'{label:20} {total:>8}'
    else:
        yield f'{label:10} {total:>9}'


_FORMATS = {
    'table': render_table,
    'full': render_full,
    'summary': render_summary,
}

### File: _files.py

In [ ]:
import os.path

from c_common.fsutil import expand_filenames, iter_files_by_suffix
from . import REPO_ROOT, INCLUDE_DIRS, SOURCE_DIRS


GLOBS = [
    'Include/*.h',
    # Technically, this is covered by "Include/*.h":
    #'Include/cpython/*.h',
    'Include/internal/*.h',
    'Include/internal/mimalloc/**/*.h',
    'Modules/**/*.h',
    'Modules/**/*.c',
    'Objects/**/*.h',
    'Objects/**/*.c',
    'Parser/**/*.h',
    'Parser/**/*.c',
    'Python/**/*.h',
    'Python/**/*.c',
]
LEVEL_GLOBS = {
    'stable': 'Include/*.h',
    'cpython': 'Include/cpython/*.h',
    'internal': 'Include/internal/*.h',
}


def resolve_filename(filename):
    orig = filename
    filename = os.path.normcase(os.path.normpath(filename))
    if os.path.isabs(filename):
        if os.path.relpath(filename, REPO_ROOT).startswith('.'):
            raise Exception(f'{orig!r} is outside the repo ({REPO_ROOT})')
        return filename
    else:
        return os.path.join(REPO_ROOT, filename)


def iter_filenames(*, search=False):
    if search:
        yield from iter_files_by_suffix(INCLUDE_DIRS, ('.h',))
        yield from iter_files_by_suffix(SOURCE_DIRS, ('.c',))
    else:
        globs = (os.path.join(REPO_ROOT, file) for file in GLOBS)
        yield from expand_filenames(globs)


def iter_header_files(filenames=None, *, levels=None):
    if not filenames:
        if levels:
            levels = set(levels)
            if 'private' in levels:
                levels.add('stable')
                levels.add('cpython')
            for level, glob in LEVEL_GLOBS.items():
                if level in levels:
                    yield from expand_filenames([glob])
        else:
            yield from iter_files_by_suffix(INCLUDE_DIRS, ('.h',))
        return

    for filename in filenames:
        orig = filename
        filename = resolve_filename(filename)
        if filename.endswith(os.path.sep):
            yield from iter_files_by_suffix(INCLUDE_DIRS, ('.h',))
        elif filename.endswith('.h'):
            yield filename
        else:
            # XXX Log it and continue instead?
            raise ValueError(f'expected .h file, got {orig!r}')

### File: _parser.py

In [ ]:
import os.path

from c_parser.preprocessor import (
    get_preprocessor as _get_preprocessor,
)
from c_parser import (
    parse_file as _parse_file,
    parse_files as _parse_files,
)
from . import REPO_ROOT


GLOB_ALL = '**/*'


def _abs(relfile):
    return os.path.join(REPO_ROOT, relfile)


def format_conf_lines(lines):
    """Format conf entries for use in a TSV table."""
    return [_abs(entry) for entry in lines]


def format_tsv_lines(lines):
    """Format entries for use in a TSV table."""
    lines = ((_abs(first), *rest) for first, *rest in lines)
    lines = map('\t'.join, lines)
    return list(map(str.rstrip, lines))


'''
@begin=sh@
./python ../c-parser/cpython.py
    --exclude '+../c-parser/EXCLUDED'
    --macros '+../c-parser/MACROS'
    --incldirs '+../c-parser/INCL_DIRS'
    --same './Include/cpython/'
    Include/*.h
    Include/internal/*.h
    Modules/**/*.c
    Objects/**/*.c
    Parser/**/*.c
    Python/**/*.c
@end=sh@
'''

# XXX Handle these.
EXCLUDED = format_conf_lines([
    # macOS
    'Modules/_scproxy.c',              # SystemConfiguration/SystemConfiguration.h

    # Windows
    'Modules/_winapi.c',               # windows.h
    'Modules/expat/winconfig.h',
    'Modules/overlapped.c',            # winsock.h
    'Python/dynload_win.c',            # windows.h
    'Python/thread_nt.h',

    # other OS-dependent
    'Python/dynload_aix.c',            # sys/ldr.h
    'Python/dynload_dl.c',             # dl.h
    'Python/dynload_hpux.c',           # dl.h
    'Python/emscripten_signal.c',
    'Python/emscripten_syscalls.c',
    'Python/emscripten_trampoline_inner.c',
    'Python/thread_pthread.h',
    'Python/thread_pthread_stubs.h',

    # only huge constants (safe but parsing is slow)
    'Modules/_ssl_data_*.h',
    'Modules/cjkcodecs/mappings_*.h',
    'Modules/unicodedata_db.h',
    'Modules/unicodename_db.h',
    'Objects/unicodetype_db.h',

    # generated
    'Python/deepfreeze/*.c',
    'Python/frozen_modules/*.h',
    'Python/generated_cases.c.h',
    'Python/executor_cases.c.h',
    'Python/optimizer_cases.c.h',
    'Python/opcode_targets.h',
    # XXX: Throws errors if PY_VERSION_HEX is not mocked out
    'Modules/clinic/_testclinic_depr.c.h',

    # not actually source
    'Python/bytecodes.c',
    'Python/optimizer_bytecodes.c',

    # mimalloc
    'Objects/mimalloc/*.c',
    'Include/internal/mimalloc/*.h',
    'Include/internal/mimalloc/mimalloc/*.h',
])

# XXX Fix the parser.
EXCLUDED += format_conf_lines([
    # The tool should be able to parse these...

    # The problem with xmlparse.c is that something
    # has gone wrong where # we handle "maybe inline actual"
    # in Tools/c-analyzer/c_parser/parser/_global.py.
    'Modules/expat/internal.h',
    'Modules/expat/xmlparse.c',
])

INCL_DIRS = format_tsv_lines([
    # (glob, dirname)

    ('*', '.'),
    ('*', './Include'),
    ('*', './Include/internal'),
    ('*', './Include/internal/mimalloc'),

    ('Modules/_decimal/**/*.c', 'Modules/_decimal/libmpdec'),
    ('Modules/_elementtree.c', 'Modules/expat'),
    ('Modules/_hacl/*.c', 'Modules/_hacl/include'),
    ('Modules/_hacl/*.c', 'Modules/_hacl/'),
    ('Modules/_hacl/*.h', 'Modules/_hacl/include'),
    ('Modules/_hacl/*.h', 'Modules/_hacl/'),
    ('Modules/md5module.c', 'Modules/_hacl/include'),
    ('Modules/sha1module.c', 'Modules/_hacl/include'),
    ('Modules/sha2module.c', 'Modules/_hacl/include'),
    ('Modules/sha3module.c', 'Modules/_hacl/include'),
    ('Modules/blake2module.c', 'Modules/_hacl/include'),
    ('Modules/hmacmodule.c', 'Modules/_hacl/include'),
    ('Objects/stringlib/*.h', 'Objects'),

    # possible system-installed headers, just in case
    ('Modules/_tkinter.c', '/usr/include/tcl8.6'),
    ('Modules/_uuidmodule.c', '/usr/include/uuid'),
    ('Modules/tkappinit.c', '/usr/include/tcl'),

])

INCLUDES = format_tsv_lines([
    # (glob, include)

    ('**/*.h', 'Python.h'),
    ('Include/**/*.h', 'object.h'),

    # for Py_HAVE_CONDVAR
    ('Include/internal/pycore_gil.h', 'pycore_condvar.h'),
    ('Python/thread_pthread.h', 'pycore_condvar.h'),

    # other

    ('Objects/stringlib/join.h', 'stringlib/stringdefs.h'),
    ('Objects/stringlib/ctype.h', 'stringlib/stringdefs.h'),
    ('Objects/stringlib/transmogrify.h', 'stringlib/stringdefs.h'),
    # ('Objects/stringlib/fastsearch.h', 'stringlib/stringdefs.h'),
    # ('Objects/stringlib/count.h', 'stringlib/stringdefs.h'),
    # ('Objects/stringlib/find.h', 'stringlib/stringdefs.h'),
    # ('Objects/stringlib/partition.h', 'stringlib/stringdefs.h'),
    # ('Objects/stringlib/split.h', 'stringlib/stringdefs.h'),
    ('Objects/stringlib/fastsearch.h', 'stringlib/ucs1lib.h'),
    ('Objects/stringlib/count.h', 'stringlib/ucs1lib.h'),
    ('Objects/stringlib/find.h', 'stringlib/ucs1lib.h'),
    ('Objects/stringlib/partition.h', 'stringlib/ucs1lib.h'),
    ('Objects/stringlib/split.h', 'stringlib/ucs1lib.h'),
    ('Objects/stringlib/find_max_char.h', 'Objects/stringlib/ucs1lib.h'),
    ('Objects/stringlib/count.h', 'Objects/stringlib/fastsearch.h'),
    ('Objects/stringlib/find.h', 'Objects/stringlib/fastsearch.h'),
    ('Objects/stringlib/partition.h', 'Objects/stringlib/fastsearch.h'),
    ('Objects/stringlib/replace.h', 'Objects/stringlib/fastsearch.h'),
    ('Objects/stringlib/repr.h', 'Objects/stringlib/fastsearch.h'),
    ('Objects/stringlib/split.h', 'Objects/stringlib/fastsearch.h'),
])

MACROS = format_tsv_lines([
    # (glob, name, value)

    ('Include/internal/*.h', 'Py_BUILD_CORE', '1'),
    ('Python/**/*.c', 'Py_BUILD_CORE', '1'),
    ('Python/**/*.h', 'Py_BUILD_CORE', '1'),
    ('Parser/**/*.c', 'Py_BUILD_CORE', '1'),
    ('Parser/**/*.h', 'Py_BUILD_CORE', '1'),
    ('Objects/**/*.c', 'Py_BUILD_CORE', '1'),
    ('Objects/**/*.h', 'Py_BUILD_CORE', '1'),

    ('Modules/_asynciomodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_codecsmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_collectionsmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_ctypes/_ctypes.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_ctypes/cfield.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_cursesmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_datetimemodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_functoolsmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_heapqmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_io/*.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_io/*.h', 'Py_BUILD_CORE', '1'),
    ('Modules/_localemodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_operator.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_posixsubprocess.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_sre/sre.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_threadmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_tracemalloc.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_weakref.c', 'Py_BUILD_CORE', '1'),
    ('Modules/_zoneinfo.c', 'Py_BUILD_CORE', '1'),
    ('Modules/atexitmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/cmathmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/faulthandler.c', 'Py_BUILD_CORE', '1'),
    ('Modules/gcmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/getpath.c', 'Py_BUILD_CORE', '1'),
    ('Modules/getpath_noop.c', 'Py_BUILD_CORE', '1'),
    ('Modules/itertoolsmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/main.c', 'Py_BUILD_CORE', '1'),
    ('Modules/mathmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/posixmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/sha256module.c', 'Py_BUILD_CORE', '1'),
    ('Modules/sha512module.c', 'Py_BUILD_CORE', '1'),
    ('Modules/signalmodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/symtablemodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/timemodule.c', 'Py_BUILD_CORE', '1'),
    ('Modules/unicodedata.c', 'Py_BUILD_CORE', '1'),

    ('Modules/_json.c', 'Py_BUILD_CORE_BUILTIN', '1'),
    ('Modules/_pickle.c', 'Py_BUILD_CORE_BUILTIN', '1'),
    ('Modules/_testinternalcapi.c', 'Py_BUILD_CORE_BUILTIN', '1'),

    ('Include/cpython/abstract.h', 'Py_CPYTHON_ABSTRACTOBJECT_H', '1'),
    ('Include/cpython/bytearrayobject.h', 'Py_CPYTHON_BYTEARRAYOBJECT_H', '1'),
    ('Include/cpython/bytesobject.h', 'Py_CPYTHON_BYTESOBJECT_H', '1'),
    ('Include/cpython/ceval.h', 'Py_CPYTHON_CEVAL_H', '1'),
    ('Include/cpython/code.h', 'Py_CPYTHON_CODE_H', '1'),
    ('Include/cpython/dictobject.h', 'Py_CPYTHON_DICTOBJECT_H', '1'),
    ('Include/cpython/fileobject.h', 'Py_CPYTHON_FILEOBJECT_H', '1'),
    ('Include/cpython/fileutils.h', 'Py_CPYTHON_FILEUTILS_H', '1'),
    ('Include/cpython/frameobject.h', 'Py_CPYTHON_FRAMEOBJECT_H', '1'),
    ('Include/cpython/import.h', 'Py_CPYTHON_IMPORT_H', '1'),
    ('Include/cpython/interpreteridobject.h', 'Py_CPYTHON_INTERPRETERIDOBJECT_H', '1'),
    ('Include/cpython/listobject.h', 'Py_CPYTHON_LISTOBJECT_H', '1'),
    ('Include/cpython/methodobject.h', 'Py_CPYTHON_METHODOBJECT_H', '1'),
    ('Include/cpython/object.h', 'Py_CPYTHON_OBJECT_H', '1'),
    ('Include/cpython/objimpl.h', 'Py_CPYTHON_OBJIMPL_H', '1'),
    ('Include/cpython/pyerrors.h', 'Py_CPYTHON_ERRORS_H', '1'),
    ('Include/cpython/pylifecycle.h', 'Py_CPYTHON_PYLIFECYCLE_H', '1'),
    ('Include/cpython/pymem.h', 'Py_CPYTHON_PYMEM_H', '1'),
    ('Include/cpython/pystate.h', 'Py_CPYTHON_PYSTATE_H', '1'),
    ('Include/cpython/sysmodule.h', 'Py_CPYTHON_SYSMODULE_H', '1'),
    ('Include/cpython/traceback.h', 'Py_CPYTHON_TRACEBACK_H', '1'),
    ('Include/cpython/tupleobject.h', 'Py_CPYTHON_TUPLEOBJECT_H', '1'),
    ('Include/cpython/unicodeobject.h', 'Py_CPYTHON_UNICODEOBJECT_H', '1'),

    # implied include of <unistd.h>
    ('Include/**/*.h', '_POSIX_THREADS', '1'),
    ('Include/**/*.h', 'HAVE_PTHREAD_H', '1'),

    # from pyconfig.h
    ('Include/cpython/pthread_stubs.h', 'HAVE_PTHREAD_STUBS', '1'),
    ('Python/thread_pthread_stubs.h', 'HAVE_PTHREAD_STUBS', '1'),

    # from Objects/bytesobject.c
    ('Objects/stringlib/partition.h', 'STRINGLIB_GET_EMPTY()', 'bytes_get_empty()'),
    ('Objects/stringlib/join.h', 'STRINGLIB_MUTABLE', '0'),
    ('Objects/stringlib/partition.h', 'STRINGLIB_MUTABLE', '0'),
    ('Objects/stringlib/split.h', 'STRINGLIB_MUTABLE', '0'),
    ('Objects/stringlib/transmogrify.h', 'STRINGLIB_MUTABLE', '0'),

    # from Makefile
    ('Modules/getpath.c', 'PYTHONPATH', '1'),
    ('Modules/getpath.c', 'PREFIX', '...'),
    ('Modules/getpath.c', 'EXEC_PREFIX', '...'),
    ('Modules/getpath.c', 'VERSION', '...'),
    ('Modules/getpath.c', 'VPATH', '...'),
    ('Modules/getpath.c', 'PLATLIBDIR', '...'),
    # ('Modules/_dbmmodule.c', 'USE_GDBM_COMPAT', '1'),
    ('Modules/_dbmmodule.c', 'USE_NDBM', '1'),
    # ('Modules/_dbmmodule.c', 'USE_BERKDB', '1'),

    # See: setup.py
    ('Modules/_decimal/**/*.c', 'CONFIG_64', '1'),
    ('Modules/_decimal/**/*.c', 'ASM', '1'),
    ('Modules/expat/xmlparse.c', 'HAVE_EXPAT_CONFIG_H', '1'),
    ('Modules/expat/xmlparse.c', 'XML_POOR_ENTROPY', '1'),
    ('Modules/_dbmmodule.c', 'HAVE_GDBM_DASH_NDBM_H', '1'),

    # others
    ('Modules/_sre/sre_lib.h', 'LOCAL(type)', 'static inline type'),
    ('Modules/_sre/sre_lib.h', 'SRE(F)', 'sre_ucs2_##F'),
    ('Objects/stringlib/codecs.h', 'STRINGLIB_IS_UNICODE', '1'),
    ('Include/internal/pycore_crossinterp_data_registry.h', 'Py_CORE_CROSSINTERP_DATA_REGISTRY_H', '1'),
])

# -pthread
# -Wno-unused-result
# -Wsign-compare
# -g
# -Og
# -Wall
# -std=c99
# -Wextra
# -Wno-unused-result -Wno-unused-parameter
# -Wno-missing-field-initializers
# -Werror=implicit-function-declaration

SAME = {
    _abs('Include/*.h'): [_abs('Include/cpython/')],
    _abs('Python/ceval.c'): ['Python/generated_cases.c.h'],
}

MAX_SIZES = {
    # GLOB: (MAXTEXT, MAXLINES),
    # default: (10_000, 200)
    # First match wins.
    _abs('Modules/_ctypes/ctypes.h'): (5_000, 500),
    _abs('Modules/_datetimemodule.c'): (20_000, 300),
    _abs('Modules/_hacl/*.c'): (200_000, 500),
    _abs('Modules/posixmodule.c'): (20_000, 500),
    _abs('Modules/termios.c'): (10_000, 800),
    _abs('Modules/_testcapimodule.c'): (20_000, 400),
    _abs('Modules/expat/expat.h'): (10_000, 400),
    _abs('Objects/stringlib/unicode_format.h'): (10_000, 400),
    _abs('Objects/typeobject.c'): (380_000, 13_000),
    _abs('Python/compile.c'): (20_000, 500),
    _abs('Python/optimizer.c'): (100_000, 5_000),
    _abs('Python/parking_lot.c'): (40_000, 1000),
    _abs('Python/pylifecycle.c'): (750_000, 5000),
    _abs('Python/pystate.c'): (750_000, 5000),
    _abs('Python/initconfig.c'): (50_000, 500),

    # Generated files:
    _abs('Include/internal/pycore_opcode.h'): (10_000, 1000),
    _abs('Include/internal/pycore_global_strings.h'): (5_000, 1000),
    _abs('Include/internal/pycore_runtime_init_generated.h'): (5_000, 1000),
    _abs('Python/deepfreeze/*.c'): (20_000, 500),
    _abs('Python/frozen_modules/*.h'): (20_000, 500),
    _abs('Python/opcode_targets.h'): (10_000, 500),
    _abs('Python/stdlib_module_names.h'): (5_000, 500),

    # These large files are currently ignored (see above).
    _abs('Modules/_ssl_data_31.h'): (80_000, 10_000),
    _abs('Modules/_ssl_data_300.h'): (80_000, 10_000),
    _abs('Modules/_ssl_data_111.h'): (80_000, 10_000),
    _abs('Modules/cjkcodecs/mappings_*.h'): (160_000, 2_000),
    _abs('Modules/unicodedata_db.h'): (180_000, 3_000),
    _abs('Modules/unicodename_db.h'): (1_200_000, 15_000),
    _abs('Objects/unicodetype_db.h'): (240_000, 3_000),

    # Catch-alls:
    _abs('Include/**/*.h'): (5_000, 500),
}


def get_preprocessor(*,
                     file_macros=None,
                     file_includes=None,
                     file_incldirs=None,
                     file_same=None,
                     **kwargs
                     ):
    macros = tuple(MACROS)
    if file_macros:
        macros += tuple(file_macros)
    includes = tuple(INCLUDES)
    if file_includes:
        includes += tuple(file_includes)
    incldirs = tuple(INCL_DIRS)
    if file_incldirs:
        incldirs += tuple(file_incldirs)
    samefiles = dict(SAME)
    if file_same:
        samefiles.update(file_same)
    return _get_preprocessor(
        file_macros=macros,
        file_includes=includes,
        file_incldirs=incldirs,
        file_same=samefiles,
        **kwargs
    )


def parse_file(filename, *,
               match_kind=None,
               ignore_exc=None,
               log_err=None,
               ):
    get_file_preprocessor = get_preprocessor(
        ignore_exc=ignore_exc,
        log_err=log_err,
    )
    yield from _parse_file(
        filename,
        match_kind=match_kind,
        get_file_preprocessor=get_file_preprocessor,
        file_maxsizes=MAX_SIZES,
    )


def parse_files(filenames=None, *,
                match_kind=None,
                ignore_exc=None,
                log_err=None,
                get_file_preprocessor=None,
                **file_kwargs
                ):
    if get_file_preprocessor is None:
        get_file_preprocessor = get_preprocessor(
            ignore_exc=ignore_exc,
            log_err=log_err,
        )
    yield from _parse_files(
        filenames,
        match_kind=match_kind,
        get_file_preprocessor=get_file_preprocessor,
        file_maxsizes=MAX_SIZES,
        **file_kwargs
    )